# 推理引擎源码与前沿优化 · 第 4/8 课：SGLang RadixAttention、内存池与 Overlap Scheduling

> 状态：**未开始**  
> 源码审阅日期：2026-08-12；vLLM `8e958902eee5`；SGLang `9deb6952afa4`。

## 本课目标与通过标准

本课产出：能逐行解释 `RadixCache.match_prefix/insert/cache_*_req`、请求池与 token-to-KV pool 的引用关系，并说明 overlap event loop 的读写屏障。

通过要求：唯一代码填空题通过全部断言；Q1～Q3 都能沿源码对象给出因果链；能指出一个正确性不变量、一个性能边界和一个需要 benchmark 才能确认的结论；总分至少 8/10。

## 源码版本与阅读方法

本课程使用固定 commit 的永久链接保证行号可复现，同时链接 current docs 供核对最新变化。阅读时先画调用图和状态所有权，再进入分支；不要从大文件第一行机械顺读。

源码阅读顺序：

1. `mem_cache/radix_cache.py::RadixKey`：token ids 之外为何还有 `extra_key`、page alignment 与 EAGLE bigram view。
2. `RadixCache.match_prefix`、`_match_prefix_helper`、`_split_node`：压缩 radix edge 怎样在中间命中时分裂。
3. `insert`、`cache_finished_req`、`cache_unfinished_req`：请求 KV 怎样移交给 cache 引用。
4. `inc_lock_ref/dec_lock_ref/evict`：protected 与 evictable 计数。
5. `Scheduler.event_loop_overlap`：何时处理上一批结果、何时 launch 当前批、WAR barrier 为什么存在。

源码快照：SGLang `9deb6952afa4`。

## 核心对象

RadixAttention 用 token prefix 构造压缩 radix tree，节点 value 指向 KV pool indices。共同前缀只存一份树路径，多轮对话、few-shot 和共享 system prompt 可复用。它与 PagedAttention 不是互斥概念：前者决定跨请求如何索引/共享前缀，后者/内存池决定物理 KV 如何分页和分配。

SGLang 至少区分 request-to-token pool（请求位置到 token/KV slot 的映射）、token-to-KV allocator（物理容量）与 tree cache（共享索引和引用）。树节点不是 KV tensor 本身。

## 调用链与状态变化

新请求用 `RadixKey(token_ids, extra_key)` 调 `match_prefix`；key 先按 page size 截断，再沿压缩 edge 寻找最长匹配，必要时 split node。命中的 device indices 被请求引用并 lock；未命中尾部由 allocator 分配。请求完成/暂停时，`cache_*_req` 把已算 KV 对应的 key/value 插入树，重复前缀部分可释放或共享；eviction 只处理未锁定叶节点。

Overlap loop 在当前 GPU forward 执行时处理上一 batch 的 CPU 结果并准备下一轮。若某 batch 不支持 overlap，先同步处理上一批；支持时先 launch 当前批，再处理上一批，并用 WAR barrier 防止共享 buffer 被过早覆盖。

## 正确性条件与常见误区

Cache key 必须隔离模型/adapter/位置语义等影响 KV 的状态；源码的 `extra_key` 是 namespace 入口。只复用 page-aligned prefix，不能把尾部未满 page 当作完整可复用 KV。lock ref 为零前才能 eviction；节点 split 不能复制/丢失对应 value 范围。

Overlap 正确性依赖 stream/event，而不是 Python 代码先后看起来合理。若上轮结果处理还读某 shared buffer，本轮写入必须等待 WAR 释放。

## 当前前沿与工程取舍

Radix tree 提高共享前缀工作负载的 cache hit，但无重复前缀时会增加 hash/tree/metadata 成本。更大 page 降低树和 allocator 元数据，却降低可复用粒度。Overlap 可隐藏 CPU 开销，但增加 batch copy、future map、buffer lifetime 和不支持特性的同步 fallback；收益必须用真实 prompt tree 和并发测试。

## 具体推演

cache 中已有 `[system, tool, userA, ...]` 与 `[system, tool, userB, ...]`。新请求 `[system, tool, userC]` 可匹配共享的前两段；若 page_size=4 而 token 级公共前缀长 6，只能安全返回前 4 个 token 的 KV。若 LoRA adapter 不同，即使 token ids 相同，`extra_key` 也应阻止共享。

请先口头复述“输入 → 状态所有者 → 状态迁移 → 输出/指标”，再做练习。

## 实践任务：唯一代码填空题

实现 page-aligned、namespace-aware 的最长前缀查询。该练习是 `RadixCache.match_prefix` 的可运行语义模型，不要求实现压缩树。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def longest_cached_prefix(query_tokens, query_namespace, cached_paths, page_size):
    if page_size <= 0:
        raise ValueError("page_size must be positive")
    best = 0
    for cached_tokens, cached_namespace in cached_paths:
        if cached_namespace != query_namespace:
            continue
        common = 0
        for left, right in zip(query_tokens, cached_tokens):
            if left != right:
                break
            common += 1
        # TODO：只有完整 page 能作为可复用 prefix 返回。
        aligned = ______
        best = max(best, aligned)
    return best

cache = [
    ([1, 2, 3, 4, 5, 9], "base"),
    ([1, 2, 3, 4, 6, 7, 8, 9], "adapter-A"),
]
assert longest_cached_prefix([1, 2, 3, 4, 5, 8], "base", cache, 4) == 4
assert longest_cached_prefix([1, 2, 3, 4, 6, 7], "adapter-A", cache, 2) == 6
assert longest_cached_prefix([1, 2, 3, 4], "adapter-B", cache, 2) == 0


### 检查方法

运行断言；增加 query 比 cache 更短的用例。解释真实 radix tree 为什么还需要 split node 和引用计数。

### Q1

RadixAttention 与 PagedAttention 分别解决哪一层问题？

**你的答案：**


### Q2

为什么 `extra_key` 是正确性和安全边界，而不只是优化标签？列出至少三项应进入 namespace 的状态。

**你的答案：**


### Q3

Overlap scheduling 中为什么需要 WAR barrier？只在 Python 中先处理上一批再写下一批不行吗？

**你的答案：**


## 评分规则

- 代码 4 分：主路径 2 分、边界条件 1 分、能映射回源码对象 1 分；
- Q1～Q3 各 2 分：必须包含对象、状态变化、正确性或成本链；
- 一票否决：把论文峰值写成普遍生产结论；把源码支持写成所有模型/硬件可用；混淆算法正确性与性能；只背类名而说不清状态所有权。

## 参考资料

- [SGLang RadixCache source](https://github.com/sgl-project/sglang/blob/9deb6952afa483e38f96385a375b96f463da5303/python/sglang/srt/mem_cache/radix_cache.py#L59-L829)
- [SGLang overlap scheduler source](https://github.com/sgl-project/sglang/blob/9deb6952afa483e38f96385a375b96f463da5303/python/sglang/srt/managers/scheduler.py#L1709-L1817)
- [SGLang memory pool source](https://github.com/sgl-project/sglang/blob/9deb6952afa483e38f96385a375b96f463da5303/python/sglang/srt/mem_cache/memory_pool.py)
- [SGLang schedule policy source](https://github.com/sgl-project/sglang/blob/9deb6952afa483e38f96385a375b96f463da5303/python/sglang/srt/managers/schedule_policy.py)
- [SGLang/RadixAttention paper](https://arxiv.org/abs/2312.07104)

源码链接固定到本课审阅 commit；current docs、支持矩阵和默认参数会变化，面试或部署前必须按目标版本重新核对。